# VOOCKS Offline Worker Handoff Scaffold (DEMO)

This notebook is a standard-library-only package inspection and file packaging example. It does **not** connect to Colab services, Drive, the network, GPUs, models, FFmpeg, or the Studio database. A profile declaring a T4 is metadata only; this notebook never probes or claims hardware. It refuses non-synthetic worker profiles because no authenticated worker adapter exists yet.

Manual flow: export a Director-authorized job package from the local dashboard; place `job_package.json` here; validate it; manually place already-produced demo fixture bytes in `staged_outputs/<declared-artifact-id>`; create `result_package.json`; then import that JSON through the local dashboard. No job-provided command, script, URL, or path is executed or fetched. The local queue rechecks lease ownership and artifact bytes before accepting a result.

In [ ]:
import base64
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import uuid

JOB_ID = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,63}$")
SHA256 = re.compile(r"^[a-f0-9]{64}$")
MAX_PACKAGE_BYTES = 16 * 1024 * 1024
MAX_OUTPUT_BYTES = 8 * 1024 * 1024

def read_json(path):
    path = Path(path)
    if path.is_symlink() or not path.is_file() or path.stat().st_size > MAX_PACKAGE_BYTES:
        raise ValueError("Package must be a bounded regular local file")
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError("Package root must be an object")
    return value

def validate_job_package(package, now=None):
    now = now or datetime.now(timezone.utc)
    expected = {"package_type", "schema_version", "manifest", "manifest_sha256", "authorization", "lease", "worker_profile", "exported_at", "execution_notice"}
    if set(package) != expected or package["package_type"] != "voocks.worker.job" or type(package["schema_version"]) is not int or package["schema_version"] != 1:
        raise ValueError("Unsupported or malformed job package")
    if package["execution_notice"] != "Offline handoff only. No hardware, worker identity, or execution environment is authenticated by this package.":
        raise ValueError("Execution notice is invalid")
    manifest = package["manifest"]
    manifest_fields = {"schema_version", "job_id", "idempotency_key", "project_id", "task_id", "workload", "input_artifacts", "output_artifacts", "resources", "max_attempts", "created_at", "deadline_at", "capabilities"}
    if not isinstance(manifest, dict) or set(manifest) != manifest_fields or type(manifest["schema_version"]) is not int or manifest["schema_version"] != 1:
        raise ValueError("Unsupported manifest schema")
    for key in ("job_id", "idempotency_key", "project_id", "task_id"):
        if not isinstance(manifest[key], str) or not JOB_ID.fullmatch(manifest[key]):
            raise ValueError("Invalid manifest identifier: " + key)
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    if not SHA256.fullmatch(package["manifest_sha256"]) or hashlib.sha256(canonical).hexdigest() != package["manifest_sha256"]:
        raise ValueError("Manifest SHA-256 mismatch")
    if manifest["workload"] not in {"cpu_ffmpeg", "gpu_inference"}:
        raise ValueError("Unsupported workload")
    resource_fields = {"accelerator", "cpu_cores", "memory_mb", "min_vram_mb", "max_runtime_seconds", "max_output_bytes", "max_storage_bytes", "max_concurrent_jobs"}
    if not isinstance(manifest["resources"], dict) or set(manifest["resources"]) != resource_fields:
        raise ValueError("Missing resource/capability declarations")
    profile, auth, lease = package["worker_profile"], package["authorization"], package["lease"]
    if not isinstance(profile, dict) or not isinstance(auth, dict) or not isinstance(lease, dict):
        raise ValueError("Worker profile, authorization, and lease must be objects")
    profile_fields = {"profile_version", "worker_id", "availability", "workloads", "cpu_cores", "memory_mb", "accelerators", "vram_mb", "processing_types", "operations", "language_tags", "model_ids", "expires_at", "is_synthetic", "provider_id"}
    capability_fields = {"operations", "language_tags", "model_ids"}
    if set(profile) != profile_fields or set(manifest["capabilities"]) != capability_fields:
        raise ValueError("Worker profile or capability contract fields are invalid")
    if profile.get("worker_id") != lease.get("worker_id") or profile.get("is_synthetic") is not True or profile.get("availability") != "available" or now >= datetime.fromisoformat(profile["expires_at"]):
        raise ValueError("Only the matching, explicitly synthetic worker profile is supported")
    if not isinstance(profile.get("worker_id"), str) or not JOB_ID.fullmatch(profile["worker_id"]):
        raise ValueError("Invalid worker identity")
    if manifest["workload"] not in profile.get("workloads", []):
        raise ValueError("Worker workload is incompatible")
    resources, caps = manifest["resources"], manifest["capabilities"]
    if type(resources.get("cpu_cores")) is not int or profile.get("cpu_cores", 0) < resources["cpu_cores"] or type(resources.get("memory_mb")) is not int or profile.get("memory_mb", 0) < resources["memory_mb"]:
        raise ValueError("Worker CPU/RAM capability is insufficient")
    if resources.get("accelerator") != "none" and (resources.get("accelerator") not in profile.get("accelerators", []) or profile.get("vram_mb", 0) < resources.get("min_vram_mb", 0)):
        raise ValueError("Worker accelerator/VRAM capability is insufficient")
    for field in ("operations", "language_tags", "model_ids"):
        if not isinstance(caps.get(field), list) or not set(caps[field]).issubset(set(profile.get(field, []))):
            raise ValueError("Worker capability mismatch: " + field)
    auth_fields = {"authorization_id", "director_id", "manifest_sha256", "authorized_at", "expires_at"}
    lease_fields = {"lease_id", "worker_id", "attempt", "acquired_at", "expires_at", "heartbeat_at"}
    if set(auth) != auth_fields or auth["manifest_sha256"] != package["manifest_sha256"] or not isinstance(auth.get("authorization_id"), str) or not JOB_ID.fullmatch(auth["authorization_id"]):
        raise ValueError("Director authorization is missing or not manifest-bound")
    if set(lease) != lease_fields or not all(isinstance(lease[k], str) and JOB_ID.fullmatch(lease[k]) for k in ("lease_id", "worker_id")):
        raise ValueError("Lease identity is invalid")
    expiry = datetime.fromisoformat(lease["expires_at"])
    auth_expiry = datetime.fromisoformat(auth["expires_at"])
    authorized_at = datetime.fromisoformat(auth["authorized_at"])
    if now.tzinfo is None or authorized_at.tzinfo is None or expiry.tzinfo is None or now >= expiry or now >= auth_expiry or auth_expiry <= authorized_at or expiry > auth_expiry or now >= datetime.fromisoformat(manifest["deadline_at"]):
        raise ValueError("Authorization, lease, or job deadline has expired")
    def validate_artifact_ref(ref, output=False):
        fields = {"artifact_id", "backend", "key", "media_type", "max_bytes"} if output else {"artifact_id", "backend", "key", "sha256", "size_bytes", "media_type"}
        if not isinstance(ref, dict) or set(ref) != fields or not isinstance(ref["artifact_id"], str) or not JOB_ID.fullmatch(ref["artifact_id"]):
            raise ValueError("Invalid artifact reference fields")
        if ref["backend"] not in {"local", "google_drive"} or not isinstance(ref["key"], str) or not ref["key"] or "\\" in ref["key"] or ref["key"].startswith("/") or re.match(r"(?i)^https?://", ref["key"]) or ".." in ref["key"].split("/") or (ref["backend"] == "google_drive" and not re.fullmatch(r"[A-Za-z0-9_-]{10,200}", ref["key"])):
            raise ValueError("Unsafe artifact backend or key")
        if not isinstance(ref["media_type"], str) or "/" not in ref["media_type"]:
            raise ValueError("Invalid artifact media type")
        size_field = "max_bytes" if output else "size_bytes"
        if type(ref[size_field]) is not int or ref[size_field] <= 0 or ref[size_field] > (4 * 1024**3 if output else 2 * 1024**3):
            raise ValueError("Invalid artifact size limit")
        if not output and (not isinstance(ref["sha256"], str) or not SHA256.fullmatch(ref["sha256"])):
            raise ValueError("Invalid input artifact checksum")
    if not isinstance(manifest["input_artifacts"], list) or not isinstance(manifest["output_artifacts"], list) or not manifest["input_artifacts"] or not manifest["output_artifacts"]:
        raise ValueError("Manifest must declare input and output artifacts")
    for ref in manifest["input_artifacts"]:
        validate_artifact_ref(ref)
    for declaration in manifest["output_artifacts"]:
        validate_artifact_ref(declaration, output=True)
    if sum(ref["size_bytes"] for ref in manifest["input_artifacts"]) > 8 * 1024**3 or sum(ref["max_bytes"] for ref in manifest["output_artifacts"]) > 4 * 1024**3:
        raise ValueError("Manifest artifact totals exceed policy")
    if type(lease["attempt"]) is not int or not 1 <= lease["attempt"] <= manifest["max_attempts"]:
        raise ValueError("Lease attempt exceeds the declared retry limit")
    if not isinstance(auth.get("director_id"), str) or not JOB_ID.fullmatch(auth["director_id"]):
        raise ValueError("Invalid Director authorization identity")
    return package

job = validate_job_package(read_json("job_package.json"))
print("DEMO package valid:", job["manifest"]["job_id"], "worker:", job["lease"]["worker_id"])
print("No hardware, execution runtime, or worker identity was verified.")

## Package staged bytes (no processing)

This cell only reads explicitly staged files named by declared artifact IDs, bounds their size, hashes their bytes, and writes a result JSON package. It does not render media, invoke a model, or execute content. The synthetic provenance is copied from the validated offline worker profile.

In [ ]:
def make_result_package(job_package, staged_directory="staged_outputs", destination="result_package.json"):
    job_package = validate_job_package(job_package)
    manifest = job_package["manifest"]
    lease = job_package["lease"]
    output_dir = Path(staged_directory).resolve(strict=True)
    if not output_dir.is_dir() or Path(staged_directory).is_symlink():
        raise ValueError("Staged output location must be a real local directory")
    outputs, total = [], 0
    for declaration in manifest["output_artifacts"]:
        artifact_id = declaration["artifact_id"]
        if not JOB_ID.fullmatch(artifact_id):
            raise ValueError("Unsafe output artifact ID")
        path = output_dir / artifact_id
        if path.is_symlink() or path.resolve(strict=True).parent != output_dir or not path.is_file():
            raise ValueError("Output must be a regular file directly inside staged_outputs")
        max_bytes = min(declaration["max_bytes"], MAX_OUTPUT_BYTES)
        if path.stat().st_size > max_bytes:
            raise ValueError("Output exceeds the handoff size limit")
        content = path.read_bytes()
        if not content or len(content) > max_bytes:
            raise ValueError("Output is empty or exceeds the handoff size limit")
        total += len(content)
        outputs.append({"artifact_id": artifact_id, "backend": declaration["backend"], "key": declaration["key"], "sha256": hashlib.sha256(content).hexdigest(), "size_bytes": len(content), "media_type": declaration["media_type"], "content_base64": base64.b64encode(content).decode("ascii")})
    if total > min(manifest["resources"]["max_output_bytes"], MAX_OUTPUT_BYTES):
        raise ValueError("Aggregate output exceeds the handoff limit")
    result = {"package_type": "voocks.worker.result", "schema_version": 1, "status": "succeeded", "job_id": manifest["job_id"], "task_id": manifest["task_id"], "project_id": manifest["project_id"], "manifest_sha256": job_package["manifest_sha256"], "authorization_id": job_package["authorization"]["authorization_id"], "worker_id": lease["worker_id"], "lease_id": lease["lease_id"], "attempt": lease["attempt"], "completed_at": datetime.now(timezone.utc).isoformat(), "result_id": "handoff-result-" + uuid.uuid4().hex, "is_synthetic": True, "error_code": None, "outputs": outputs}
    destination = Path(destination)
    if destination.parent.resolve() != Path.cwd().resolve() or destination.exists() or destination.is_symlink():
        raise ValueError("Refusing to overwrite an existing result package")
    encoded = json.dumps(result, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    if len(encoded) > MAX_PACKAGE_BYTES:
        raise ValueError("Result package exceeds the handoff limit")
    with destination.open("xb") as output_file:
        output_file.write(encoded)
    print("Synthetic DEMO result package written:", destination)
    return result

def make_interruption_package(job_package, destination="result_package.json"):
    job_package = validate_job_package(job_package)
    manifest, lease = job_package["manifest"], job_package["lease"]
    result = {"package_type": "voocks.worker.result", "schema_version": 1, "status": "interrupted", "job_id": manifest["job_id"], "task_id": manifest["task_id"], "project_id": manifest["project_id"], "manifest_sha256": job_package["manifest_sha256"], "authorization_id": job_package["authorization"]["authorization_id"], "worker_id": lease["worker_id"], "lease_id": lease["lease_id"], "attempt": lease["attempt"], "completed_at": datetime.now(timezone.utc).isoformat(), "result_id": "handoff-result-" + uuid.uuid4().hex, "is_synthetic": True, "error_code": "WORKER_INTERRUPTED", "outputs": []}
    destination = Path(destination)
    if destination.parent.resolve() != Path.cwd().resolve() or destination.exists() or destination.is_symlink():
        raise ValueError("Refusing to overwrite an existing result package")
    encoded = json.dumps(result, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    if len(encoded) > MAX_PACKAGE_BYTES:
        raise ValueError("Result package exceeds the handoff limit")
    with destination.open("xb") as output_file:
        output_file.write(encoded)
    print("Synthetic interruption report written; Director retry remains required.")
    return result

# Run only after manually placing DEMO bytes in staged_outputs/<artifact_id>.
# result = make_result_package(job)